# 02 - Selecionar passagem teste SWOT

Seleciona um pequeno conjunto de candidatos SWOT para validar a observabilidade real dos 13 exutórios, usando apenas metadados NASA Earthdata/PO.DAAC. Este notebook não baixa PIXC pesado automaticamente e não faz download em massa.

## Objetivo

A etapa 01 confirmou cobertura potencial por envelope espacial com buffer. Esta etapa organiza os grânulos encontrados por produto, ciclo, passagem, tile, datas, tamanho estimado e link/identificador de acesso, para escolher uma passagem/produto de teste antes de qualquer processamento pesado.

## Cobertura potencial vs observabilidade real

Cobertura potencial significa que o catálogo encontrou produtos SWOT interceptando o envelope dos exutórios. Observabilidade real exige confirmar se um ponto está efetivamente dentro da faixa útil do produto, se há pixels/feições de água próximos, e se a qualidade do produto permite análise. Essa validação fina será feita depois com um pequeno produto de teste, não nesta etapa de censo.

In [1]:
from __future__ import annotations

import logging
import re
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
import geopandas as gpd


In [2]:
def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'src' / 'check_environment.py').exists():
            return candidate
    fallback = Path.home() / 'mystorage' / 'PPGGAG1889' / 'atividade3_swot'
    if fallback.exists():
        return fallback
    raise RuntimeError('FALHA: rode este notebook dentro do repositorio atividade3_swot.')

PROJECT_ROOT = find_project_root(Path.cwd())
EXUTORIOS_CSV = PROJECT_ROOT / 'dados' / 'exutorios.csv'
OBSERVABILIDADE_CSV = PROJECT_ROOT / 'outputs' / 'tabelas' / 'observabilidade_exutorios.csv'
OUTPUT_CANDIDATES = PROJECT_ROOT / 'outputs' / 'tabelas' / 'candidatos_passagem_teste_swot.csv'
LOG_FILE = PROJECT_ROOT / 'outputs' / 'logs' / '02_selecionar_passagem_teste_swot.log'

OUTPUT_CANDIDATES.parent.mkdir(parents=True, exist_ok=True)
LOG_FILE.parent.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    filename=LOG_FILE,
    filemode='w',
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(message)s',
)

print('OK raiz do projeto:', PROJECT_ROOT)
print('OK log:', LOG_FILE)


OK raiz do projeto: /home/jovyan/mystorage/PPGGAG1889/atividade3_swot
OK log: /home/jovyan/mystorage/PPGGAG1889/atividade3_swot/outputs/logs/02_selecionar_passagem_teste_swot.log


## Entradas

O notebook lê `dados/exutorios.csv` e, quando disponível, `outputs/tabelas/observabilidade_exutorios.csv`. A tabela de observabilidade da etapa 01 é usada como contexto, mas a seleção de candidatos consulta novamente o catálogo para obter metadados completos.

In [3]:
expected_columns = ['id', 'latitude', 'longitude']
if not EXUTORIOS_CSV.exists():
    raise FileNotFoundError(f'FALHA: arquivo de exutorios nao encontrado: {EXUTORIOS_CSV}')

exutorios = pd.read_csv(EXUTORIOS_CSV)
if list(exutorios.columns) != expected_columns:
    raise ValueError(f'FALHA: colunas esperadas {expected_columns}, colunas encontradas {list(exutorios.columns)}')
if len(exutorios) != 13:
    raise ValueError(f'FALHA: esperados 13 exutorios, encontrados {len(exutorios)}')

exutorios['latitude'] = pd.to_numeric(exutorios['latitude'], errors='raise')
exutorios['longitude'] = pd.to_numeric(exutorios['longitude'], errors='raise')

observabilidade = None
if OBSERVABILIDADE_CSV.exists():
    observabilidade = pd.read_csv(OBSERVABILIDADE_CSV)
    logging.info('Tabela de observabilidade lida: %s', OBSERVABILIDADE_CSV)
    print('OK observabilidade anterior:', OBSERVABILIDADE_CSV)
else:
    logging.warning('Tabela de observabilidade anterior nao encontrada: %s', OBSERVABILIDADE_CSV)
    print('AVISO observabilidade anterior nao encontrada:', OBSERVABILIDADE_CSV)

logging.info('Exutorios lidos: %s linhas', len(exutorios))
print('OK exutorios:', EXUTORIOS_CSV)
display(exutorios)


OK observabilidade anterior: /home/jovyan/mystorage/PPGGAG1889/atividade3_swot/outputs/tabelas/observabilidade_exutorios.csv
OK exutorios: /home/jovyan/mystorage/PPGGAG1889/atividade3_swot/dados/exutorios.csv


,id,latitude,longitude
0,P01,-15.539765,-47.972069
1,P02,-15.541334,-47.969589
2,P03,-15.541921,-47.981644
3,P04,-15.542892,-47.980763
4,P05,-15.542658,-47.979778
5,P06,-15.543083,-47.979286
6,P07,-15.541673,-47.979366
7,P08,-15.537334,-47.975750
8,P09,-15.537017,-47.975281
9,P10,-15.536032,-47.976860


## Geometria e envelope com buffer

A geometria reproduz a lógica do notebook 01: pontos em `EPSG:4326`, reprojeção automática para um CRS métrico local, buffer de segurança e retorno do envelope para WGS84. A consulta ao CMR usa esse `bounding_box`.

In [4]:
BUFFER_METERS = 5000

gdf = gpd.GeoDataFrame(
    exutorios.copy(),
    geometry=gpd.points_from_xy(exutorios['longitude'], exutorios['latitude']),
    crs='EPSG:4326',
)

utm_crs = gdf.estimate_utm_crs()
if utm_crs is None:
    raise RuntimeError('FALHA: nao foi possivel estimar CRS metrico para o buffer.')

gdf_m = gdf.to_crs(utm_crs)
buffered_union = gdf_m.geometry.buffer(BUFFER_METERS).union_all()
bbox_wgs84 = gpd.GeoSeries([buffered_union.envelope], crs=utm_crs).to_crs('EPSG:4326').iloc[0]
bbox = tuple(bbox_wgs84.bounds)

logging.info('CRS metrico estimado: %s', utm_crs)
logging.info('Buffer aplicado: %s metros', BUFFER_METERS)
logging.info('Bounding box WGS84: %s', bbox)
print('OK CRS metrico:', utm_crs)
print('OK buffer_m:', BUFFER_METERS)
print('OK bounding_box:', bbox)
display(gdf)


OK CRS metrico: EPSG:32723
OK buffer_m: 5000
OK bounding_box: (-48.02893635512781, -15.59256934663154, -47.915784197830234, -15.490193219201679)


,id,latitude,longitude,geometry
0,P01,-15.539765,-47.972069,POINT (-47.97207 -15.53976)
1,P02,-15.541334,-47.969589,POINT (-47.96959 -15.54133)
2,P03,-15.541921,-47.981644,POINT (-47.98164 -15.54192)
3,P04,-15.542892,-47.980763,POINT (-47.98076 -15.54289)
4,P05,-15.542658,-47.979778,POINT (-47.97978 -15.54266)
5,P06,-15.543083,-47.979286,POINT (-47.97929 -15.54308)
6,P07,-15.541673,-47.979366,POINT (-47.97937 -15.54167)
7,P08,-15.537334,-47.975750,POINT (-47.97575 -15.53733)
8,P09,-15.537017,-47.975281,POINT (-47.97528 -15.53702)
9,P10,-15.536032,-47.976860,POINT (-47.97686 -15.53603)


## Critérios de ranqueamento

Os candidatos são priorizados assim:

1. produtos mais leves antes de PIXC pesado quando úteis para inspeção preliminar: RiverSP, depois LakeSP, PIXC apenas como referência;
2. grânulos retornados pela consulta espacial do envelope com buffer;
3. datas mais recentes;
4. presença de data inicial/final, tamanho estimado e URL de download;
5. metadados com cycle/pass/tile identificáveis no nome do grânulo.

A saída final seleciona automaticamente um candidato RiverSP, um LakeSP e um PIXC de referência, sem baixar PIXC.

In [5]:
try:
    import earthaccess
except Exception as exc:
    logging.exception('Falha ao importar earthaccess')
    raise RuntimeError('FALHA: earthaccess nao esta instalado. Rode pip install -r requirements.txt.') from exc

START_DATE = '2023-01-01'
END_DATE = datetime.now(timezone.utc).date().isoformat()
PRODUCTS = {
    'SWOT_L2_HR_RIVERSP_D': {'pattern': '*Reach*', 'produto': 'RIVERSP', 'peso_rank': 1},
    'SWOT_L2_HR_LAKESP_D': {'pattern': '*Prior*', 'produto': 'LAKESP', 'peso_rank': 2},
    'SWOT_L2_HR_PIXC_D': {'pattern': '*', 'produto': 'PIXC', 'peso_rank': 3},
}

def granule_umm(granule) -> dict:
    if hasattr(granule, 'umm'):
        return granule.umm
    if isinstance(granule, dict):
        return granule.get('umm', granule)
    try:
        return granule['umm']
    except Exception:
        return {}

def granule_native_id(granule) -> str:
    umm = granule_umm(granule)
    return umm.get('GranuleUR') or umm.get('ProducerGranuleId') or str(granule)

def get_time_bounds(umm: dict, name: str):
    temporal = umm.get('TemporalExtent', {}) or {}
    range_time = temporal.get('RangeDateTime', {}) or {}
    start = pd.to_datetime(range_time.get('BeginningDateTime'), errors='coerce')
    end = pd.to_datetime(range_time.get('EndingDateTime'), errors='coerce')
    if pd.isna(start):
        match = re.search(r'_(20\d{6}T\d{6})_', name)
        if match:
            start = pd.to_datetime(match.group(1), format='%Y%m%dT%H%M%S', errors='coerce')
    if pd.isna(end):
        matches = re.findall(r'(20\d{6}T\d{6})', name)
        if len(matches) >= 2:
            end = pd.to_datetime(matches[1], format='%Y%m%dT%H%M%S', errors='coerce')
    return start, end

def parse_cycle_pass_tile(name: str) -> dict:
    patterns = [
        r'SWOT_L2_HR_PIXC_(\d{3})_(\d{3})_([0-9]{3}[LR])_',
        r'SWOT_L2_HR_RiverSP_Reach_(\d{3})_(\d{3})_([A-Z]{2})_',
        r'SWOT_L2_HR_LakeSP_Prior_(\d{3})_(\d{3})_([A-Z]{2})_',
    ]
    for pattern in patterns:
        match = re.search(pattern, name)
        if match:
            return {'cycle': match.group(1), 'pass': match.group(2), 'tile': match.group(3)}
    return {'cycle': None, 'pass': None, 'tile': None}

def size_mb(umm: dict):
    value = umm.get('DataGranule', {}).get('ArchiveAndDistributionInformation', [])
    sizes = []
    for item in value or []:
        size = item.get('Size')
        unit = str(item.get('SizeUnit', 'MB')).upper()
        try:
            size = float(size)
        except Exception:
            continue
        if unit in {'KB', 'KBYTES', 'KBYTE'}:
            size = size / 1024
        elif unit in {'GB', 'GBYTES', 'GBYTE'}:
            size = size * 1024
        elif unit in {'BYTES', 'BYTE', 'B'}:
            size = size / (1024 * 1024)
        sizes.append(size)
    return max(sizes) if sizes else None

def download_url(umm: dict) -> str:
    urls = []
    for item in umm.get('RelatedUrls', []) or []:
        url = item.get('URL')
        if not url:
            continue
        url_type = str(item.get('Type', '')).upper()
        subtype = str(item.get('Subtype', '')).upper()
        urls.append((url_type, subtype, url))
    preferred = [u for t, s, u in urls if 'GET DATA' in t or 'DOWNLOAD' in s or 'DATA' in t]
    if preferred:
        return preferred[0]
    return urls[0][2] if urls else ''

records = []
for short_name, cfg in PRODUCTS.items():
    logging.info('Consultando %s', short_name)
    try:
        results = earthaccess.search_data(
            short_name=short_name,
            temporal=(START_DATE, END_DATE),
            bounding_box=bbox,
            granule_name=cfg['pattern'],
            count=-1,
        )
    except Exception as exc:
        logging.exception('Falha na consulta %s', short_name)
        raise RuntimeError(
            f'FALHA na consulta Earthdata/CMR para {short_name}. '
            'Verifique internet, pacote earthaccess e, se necessario, login Earthdata manual.'
        ) from exc

    print(f'OK consulta {short_name}: {len(results)} granulos')
    logging.info('%s: %s granulos encontrados', short_name, len(results))
    for granule in results:
        umm = granule_umm(granule)
        granule_id = granule_native_id(granule)
        start, end = get_time_bounds(umm, granule_id)
        parsed = parse_cycle_pass_tile(granule_id)
        url = download_url(umm)
        records.append({
            'produto': cfg['produto'],
            'short_name': short_name,
            'cycle': parsed['cycle'],
            'pass': parsed['pass'],
            'tile': parsed['tile'],
            'data_inicio': start,
            'data_fim': end,
            'tamanho_mb': size_mb(umm),
            'granule_id': granule_id,
            'download_url': url,
            'peso_produto': cfg['peso_rank'],
            'tem_url': bool(url),
            'tem_tempo': pd.notna(start),
            'tem_cycle_pass': parsed['cycle'] is not None and parsed['pass'] is not None,
        })

metadados = pd.DataFrame.from_records(records)
if metadados.empty:
    raise RuntimeError('FALHA: nenhum metadado SWOT encontrado para a area consultada.')

metadados['data_inicio'] = pd.to_datetime(metadados['data_inicio'], errors='coerce', utc=True)
metadados['data_fim'] = pd.to_datetime(metadados['data_fim'], errors='coerce', utc=True)
metadados['tamanho_mb'] = pd.to_numeric(metadados['tamanho_mb'], errors='coerce')
metadados = metadados.drop_duplicates(subset=['produto', 'granule_id']).reset_index(drop=True)

print('Total de metadados unicos:', len(metadados))
display(metadados.sort_values(['produto', 'data_inicio'], ascending=[True, False]).head(20))


/opt/conda/lib/python3.11/site-packages/earthaccess/results.py:343: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()


OK consulta SWOT_L2_HR_RIVERSP_D: 472 granulos
OK consulta SWOT_L2_HR_LAKESP_D: 478 granulos
OK consulta SWOT_L2_HR_PIXC_D: 216 granulos
Total de metadados unicos: 1166


,produto,short_name,cycle,pass,tile,data_inicio,data_fim,tamanho_mb,granule_id,download_url,peso_produto,tem_url,tem_tempo,tem_cycle_pass
949,LAKESP,SWOT_L2_HR_LAKESP_D,055,255,SA,2026-08-30 00:11:07.439000+00:00,2026-08-30 00:21:21.989000+00:00,12.050320,SWOT_L2_HR_LakeSP_Prior_055_255_SA_20260830T00...,https://archive.swot.podaac.earthdata.nasa.gov...,2,True,True,True
948,LAKESP,SWOT_L2_HR_LAKESP_D,055,227,SA,2026-08-29 00:11:40.006000+00:00,2026-08-29 00:20:36.006000+00:00,8.882739,SWOT_L2_HR_LakeSP_Prior_055_227_SA_20260829T00...,https://archive.swot.podaac.earthdata.nasa.gov...,2,True,True,True
946,LAKESP,SWOT_L2_HR_LAKESP_D,055,158,SA,2026-08-26 13:09:17.158000+00:00,2026-08-26 13:18:37.874000+00:00,8.684681,SWOT_L2_HR_LakeSP_Prior_055_158_SA_20260826T13...,https://archive.swot.podaac.earthdata.nasa.gov...,2,True,True,True
947,LAKESP,SWOT_L2_HR_LAKESP_D,055,158,SA,2026-08-26 13:09:17.158000+00:00,2026-08-26 13:18:37.874000+00:00,8.688951,SWOT_L2_HR_LakeSP_Prior_055_158_SA_20260826T13...,https://archive.swot.podaac.earthdata.nasa.gov...,2,True,True,True
945,LAKESP,SWOT_L2_HR_LAKESP_D,055,130,SA,2026-08-25 13:09:44.569000+00:00,2026-08-25 13:17:26.326000+00:00,12.222147,SWOT_L2_HR_LakeSP_Prior_055_130_SA_20260825T13...,https://archive.swot.podaac.earthdata.nasa.gov...,2,True,True,True
944,LAKESP,SWOT_L2_HR_LAKESP_D,054,533,SA,2026-08-19 01:49:02.597000+00:00,2026-08-19 01:58:23.507000+00:00,15.271619,SWOT_L2_HR_LakeSP_Prior_054_533_SA_20260819T01...,https://archive.swot.podaac.earthdata.nasa.gov...,2,True,True,True
943,LAKESP,SWOT_L2_HR_LAKESP_D,054,505,SA,2026-08-18 01:49:31.748000+00:00,2026-08-18 01:57:38.910000+00:00,6.295926,SWOT_L2_HR_LakeSP_Prior_054_505_SA_20260818T01...,https://archive.swot.podaac.earthdata.nasa.gov...,2,True,True,True
942,LAKESP,SWOT_L2_HR_LAKESP_D,054,464,SA,2026-08-16 14:46:52.451000+00:00,2026-08-16 14:57:03.667000+00:00,9.351173,SWOT_L2_HR_LakeSP_Prior_054_464_SA_20260816T14...,https://archive.swot.podaac.earthdata.nasa.gov...,2,True,True,True
941,LAKESP,SWOT_L2_HR_LAKESP_D,054,436,SA,2026-08-15 14:46:47.513000+00:00,2026-08-15 14:55:21.928000+00:00,12.108055,SWOT_L2_HR_LakeSP_Prior_054_436_SA_20260815T14...,https://archive.swot.podaac.earthdata.nasa.gov...,2,True,True,True
939,LAKESP,SWOT_L2_HR_LAKESP_D,054,408,SA,2026-08-14 14:47:36.525000+00:00,2026-08-14 14:54:31.289000+00:00,5.056863,SWOT_L2_HR_LakeSP_Prior_054_408_SA_20260814T14...,https://archive.swot.podaac.earthdata.nasa.gov...,2,True,True,True


In [6]:
def rank_candidates(group: pd.DataFrame) -> pd.DataFrame:
    return group.sort_values(
        by=['tem_tempo', 'tem_url', 'tem_cycle_pass', 'data_inicio', 'tamanho_mb'],
        ascending=[False, False, False, False, True],
        na_position='last',
    )

selected = []
for produto in ['RIVERSP', 'LAKESP', 'PIXC']:
    subset = metadados[metadados['produto'] == produto].copy()
    if subset.empty:
        logging.warning('Nenhum candidato para %s', produto)
        continue
    best = rank_candidates(subset).iloc[0].copy()
    if produto == 'PIXC':
        motivo = 'Melhor PIXC recente identificado apenas como referencia; nao baixar automaticamente por ser potencialmente pesado.'
    else:
        motivo = f'Melhor candidato {produto} recente no envelope com buffer; produto vetorial/mais leve para validacao preliminar.'
    best['motivo_rank'] = motivo
    selected.append(best)

candidatos = pd.DataFrame(selected)
if candidatos.empty:
    raise RuntimeError('FALHA: nenhum candidato selecionado.')

candidatos = candidatos.sort_values(['peso_produto', 'data_inicio'], ascending=[True, False]).reset_index(drop=True)
candidatos.insert(0, 'rank', range(1, len(candidatos) + 1))

output_columns = [
    'rank', 'produto', 'cycle', 'pass', 'tile', 'data_inicio', 'data_fim',
    'tamanho_mb', 'granule_id', 'download_url', 'motivo_rank'
]
candidatos_out = candidatos[output_columns].copy()
for col in ['data_inicio', 'data_fim']:
    candidatos_out[col] = candidatos_out[col].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

candidatos_out.to_csv(OUTPUT_CANDIDATES, index=False, encoding='utf-8')

with LOG_FILE.open('a', encoding='utf-8') as log:
    log.write('\nResumo da selecao de passagem teste SWOT\n')
    log.write(f'Area bbox WGS84: {bbox}\n')
    log.write(f'Total metadados unicos: {len(metadados)}\n')
    for produto, count in metadados['produto'].value_counts().sort_index().items():
        log.write(f'{produto}: {count} granulos\n')
    log.write(f'Tabela de candidatos: {OUTPUT_CANDIDATES}\n')
    log.write('Nenhum download foi executado neste notebook.\n')

print('OK candidatos salvos:', OUTPUT_CANDIDATES)
print('OK log atualizado:', LOG_FILE)
display(candidatos_out)


OK candidatos salvos: /home/jovyan/mystorage/PPGGAG1889/atividade3_swot/outputs/tabelas/candidatos_passagem_teste_swot.csv
OK log atualizado: /home/jovyan/mystorage/PPGGAG1889/atividade3_swot/outputs/logs/02_selecionar_passagem_teste_swot.log


,rank,produto,cycle,pass,tile,data_inicio,data_fim,tamanho_mb,granule_id,download_url,motivo_rank
0,1,RIVERSP,055,255,SA,2026-08-30T00:11:07Z,2026-08-30T00:21:21Z,10.732728,SWOT_L2_HR_RiverSP_Reach_055_255_SA_20260830T0...,https://archive.swot.podaac.earthdata.nasa.gov...,Melhor candidato RIVERSP recente no envelope c...
1,2,LAKESP,055,255,SA,2026-08-30T00:11:07Z,2026-08-30T00:21:21Z,12.050320,SWOT_L2_HR_LakeSP_Prior_055_255_SA_20260830T00...,https://archive.swot.podaac.earthdata.nasa.gov...,Melhor candidato LAKESP recente no envelope co...
2,3,PIXC,055,227,127L,2026-08-29T00:16:10Z,2026-08-29T00:16:21Z,311.331682,SWOT_L2_HR_PIXC_055_227_127L_20260829T001610_2...,https://archive.swot.podaac.earthdata.nasa.gov...,Melhor PIXC recente identificado apenas como r...


## Limitações

- A seleção ainda é baseada no envelope com buffer e nos metadados retornados pelo catálogo.
- A presença de um grânulo no catálogo não garante observação válida exatamente no ponto do exutório.
- O candidato PIXC é listado apenas como referência, sem download automático.
- Tamanho e URLs dependem da completude dos metadados CMR retornados.
- A observabilidade real exigirá abrir um produto de teste e verificar geometrias/pixels/qualidade.


## Próximos passos

1. Revisar `outputs/tabelas/candidatos_passagem_teste_swot.csv`.
2. Escolher preferencialmente o candidato RiverSP ou LakeSP mais leve para validação inicial.
3. Criar uma etapa de download controlado de no máximo um produto pequeno.
4. Só considerar PIXC depois, quando for necessário validar pixels e com cuidado para evitar download pesado.
